# Selección de datos: corredor transcontinental de EE. UU.

## 1. Objetivo

Este notebook genera un subconjunto reproducible del dataset original `itineraries.csv`, conservando las 27 columnas originales y seleccionando vuelos entre los aeropuertos del oeste **LAX/SFO** y los del este **JFK/EWR/BOS**, en ambos sentidos.

El resultado se utilizará en las siguientes fases del proyecto para estudiar si el precio de un vuelo puede bajar al menos un **5 % durante los tres días siguientes**. Aquí no se construye todavía la variable objetivo ni se eliminan columnas.

**Entrada:** `data/itineraries.csv`  
**Salida:** `data/corredor_transcontinental.csv`

> La salida es un CSV único. Spark escribe los CSV como directorios con archivos `part-*`, por lo que al final se moverá el único archivo generado a la ruta de salida indicada.

## 2. Configuración del proyecto

El notebook se encuentra dentro de `notebooks/`, por lo que se toma como raíz del proyecto la carpeta superior. Si tu estructura de carpetas es diferente, modifica `PROJECT_DIR` en la celda siguiente.

La configuración de Java y Hadoop está adaptada al entorno Windows donde se ha probado Spark. Si las variables de entorno ya están configuradas de forma permanente, puedes conservarlas o ajustar las rutas.

In [1]:
import os
import shutil
from pathlib import Path

# Rutas del proyecto: el notebook está en la carpeta notebooks/
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
CSV_PATH = DATA_DIR / "itineraries.csv"
OUTPUT_PATH = DATA_DIR / "corredor_transcontinental.csv"
TEMP_OUTPUT_DIR = DATA_DIR / "_corredor_transcontinental_csv_tmp"

# Entorno Windows (ajustar solo si tus rutas son distintas)
JAVA_PATH = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
HADOOP_PATH = r"C:\hadoop"

if os.name == "nt":
    if Path(JAVA_PATH).exists():
        os.environ["JAVA_HOME"] = JAVA_PATH
        os.environ["PATH"] = str(Path(JAVA_PATH) / "bin") + os.pathsep + os.environ.get("PATH", "")
    if Path(HADOOP_PATH).exists():
        os.environ["HADOOP_HOME"] = HADOOP_PATH
        os.environ["PATH"] = str(Path(HADOOP_PATH) / "bin") + os.pathsep + os.environ.get("PATH", "")
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Raíz del proyecto:", PROJECT_DIR)
print("CSV original:", CSV_PATH)
print("CSV de salida:", OUTPUT_PATH)
print("Existe el CSV original:", CSV_PATH.exists())

Raíz del proyecto: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1
CSV original: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1\data\itineraries.csv
CSV de salida: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1\data\corredor_transcontinental.csv
Existe el CSV original: True


## 3. Inicialización de Spark

Se utiliza Spark en modo local para ejecutar el filtrado en el equipo. Se configura la dirección local para evitar problemas de resolución del nombre del equipo en Windows.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"No se encuentra el dataset original: {CSV_PATH}. "
        "Comprueba la carpeta data/ y la variable PROJECT_DIR."
    )

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("AAGE_Data_Selection")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 4.0.1


## 4. Lectura del dataset original

Se desactiva la inferencia automática de tipos para evitar que Spark tenga que recorrer el archivo con ese propósito y para conservar los valores originales durante esta fase de selección. Los tipos se analizarán y convertirán cuando sean necesarios en el preprocesamiento.

In [3]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(str(CSV_PATH))
)

print("Número de columnas:", len(df.columns))
print("Columnas:")
print(df.columns)

df.select(
    "searchDate", "flightDate", "startingAirport",
    "destinationAirport", "totalFare"
).show(5, truncate=False)

Número de columnas: 27
Columnas:
['legId', 'searchDate', 'flightDate', 'startingAirport', 'destinationAirport', 'fareBasisCode', 'travelDuration', 'elapsedDays', 'isBasicEconomy', 'isRefundable', 'isNonStop', 'baseFare', 'totalFare', 'seatsRemaining', 'totalTravelDistance', 'segmentsDepartureTimeEpochSeconds', 'segmentsDepartureTimeRaw', 'segmentsArrivalTimeEpochSeconds', 'segmentsArrivalTimeRaw', 'segmentsArrivalAirportCode', 'segmentsDepartureAirportCode', 'segmentsAirlineName', 'segmentsAirlineCode', 'segmentsEquipmentDescription', 'segmentsDurationInSeconds', 'segmentsDistance', 'segmentsCabinCode']
+----------+----------+---------------+------------------+---------+
|searchDate|flightDate|startingAirport|destinationAirport|totalFare|
+----------+----------+---------------+------------------+---------+
|2022-04-16|2022-04-17|ATL            |BOS               |248.60   |
|2022-04-16|2022-04-17|ATL            |BOS               |248.60   |
|2022-04-16|2022-04-17|ATL            |BOS  

## 5. Selección de las rutas

Se consideran tres aeropuertos del oeste (Los Ángeles y San Francisco) y tres del este (Nueva York-JFK, Newark y Boston). Se incluyen los dos sentidos de viaje, por lo que se esperan **12 rutas dirigidas**.

No se filtra por fecha ni se seleccionan columnas en esta fase: el objetivo es preservar todas las observaciones disponibles para estas rutas.

In [4]:
aeropuertos_oeste = ["LAX", "SFO"]
aeropuertos_este = ["JFK", "EWR", "BOS"]

ida = (
    F.col("startingAirport").isin(aeropuertos_oeste)
    & F.col("destinationAirport").isin(aeropuertos_este)
)
vuelta = (
    F.col("startingAirport").isin(aeropuertos_este)
    & F.col("destinationAirport").isin(aeropuertos_oeste)
)

df_subset = df.filter(ida | vuelta)

print("Filtro de rutas aplicado.")

Filtro de rutas aplicado.


## 6. Validación del subconjunto

Se comprueban las dimensiones del resultado y el número de registros de cada ruta. Estos cálculos recorren los datos, pero sirven para verificar que el filtrado ha producido el subconjunto esperado antes de exportarlo.

In [5]:
print("Número de columnas:", len(df_subset.columns))
print("Número de filas:", df_subset.count())

print("\nNúmero de filas por ruta dirigida:")
rutas = (
    df_subset
    .groupBy("startingAirport", "destinationAirport")
    .count()
    .orderBy("startingAirport", "destinationAirport")
)
rutas.show(20, truncate=False)

print("Número de rutas dirigidas presentes:", rutas.count())
print("Número de legId únicos:", df_subset.select("legId").distinct().count())

assert len(df_subset.columns) == 27, "El número de columnas no coincide con el esperado."
assert rutas.count() == 12, "No se han encontrado las 12 rutas dirigidas esperadas."

Número de columnas: 27
Número de filas: 6147167

Número de filas por ruta dirigida:
+---------------+------------------+------+
|startingAirport|destinationAirport|count |
+---------------+------------------+------+
|BOS            |LAX               |644390|
|BOS            |SFO               |485993|
|EWR            |LAX               |544467|
|EWR            |SFO               |366756|
|JFK            |LAX               |605017|
|JFK            |SFO               |355357|
|LAX            |BOS               |679169|
|LAX            |EWR               |587270|
|LAX            |JFK               |625496|
|SFO            |BOS               |507030|
|SFO            |EWR               |400532|
|SFO            |JFK               |345690|
+---------------+------------------+------+

Número de rutas dirigidas presentes: 12
Número de legId únicos: 518540


## 7. Exportación a un único CSV

Spark normalmente escribe una carpeta con uno o varios archivos `part-*`. Para facilitar el uso del resultado en otros notebooks y herramientas, se genera una única partición de salida y se mueve el archivo CSV resultante a `data/corredor_transcontinental.csv`.

**Nota:** `coalesce(1)` concentra la escritura en una sola tarea. Es adecuado para este subconjunto de tamaño moderado, pero no sería recomendable para exportaciones masivas mucho mayores.

In [6]:
# Limpiar posibles restos de ejecuciones anteriores
if TEMP_OUTPUT_DIR.exists():
    shutil.rmtree(TEMP_OUTPUT_DIR)
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

# Spark escribe un directorio con un único archivo part-*.csv
(
    df_subset
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(str(TEMP_OUTPUT_DIR))
)

archivos_csv = list(TEMP_OUTPUT_DIR.glob("part-*.csv"))
if len(archivos_csv) != 1:
    raise RuntimeError(
        f"Se esperaba un único archivo part-*.csv, pero se encontraron {len(archivos_csv)}."
    )

shutil.move(str(archivos_csv[0]), str(OUTPUT_PATH))
shutil.rmtree(TEMP_OUTPUT_DIR)

print("CSV guardado correctamente:", OUTPUT_PATH)
print(f"Tamaño del CSV: {OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

CSV guardado correctamente: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1\data\corredor_transcontinental.csv
Tamaño del CSV: 2085.94 MB


## 8. Resumen final

Esta sección deja registrado el resultado de la selección para facilitar la reproducibilidad del proyecto.

In [7]:
print("RESUMEN DE LA SELECCIÓN")
print("-" * 40)
print(f"Archivo original: {CSV_PATH}")
print(f"Archivo seleccionado: {OUTPUT_PATH}")
print(f"Columnas conservadas: {len(df_subset.columns)}")
print(f"Filas seleccionadas: {df_subset.count():,}")
print(f"Rutas dirigidas: {rutas.count()}")
print(f"Tamaño CSV: {OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

print("\nEl CSV conserva las columnas originales; la variable objetivo se construirá en una fase posterior.")

RESUMEN DE LA SELECCIÓN
----------------------------------------
Archivo original: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1\data\itineraries.csv
Archivo seleccionado: c:\UNI\Q7\AAGE\Prácticas\Práctica_1.1\data\corredor_transcontinental.csv
Columnas conservadas: 27
Filas seleccionadas: 6,147,167
Rutas dirigidas: 12
Tamaño CSV: 2085.94 MB

El CSV conserva las columnas originales; la variable objetivo se construirá en una fase posterior.
